# Text prediction/generation with Keras using LSTM

   In this example we will work with the book: Alice’s Adventures in Wonderland by Lewis Carroll.

  We are going to learn the dependencies between characters and the conditional probabilities of characters in sequences so that we can in turn generate wholly new and original sequences of characters.

In [1]:
import numpy as np

import tensorflow as tf
import keras

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import LSTM
from keras.callbacks import ModelCheckpoint
from keras.utils import np_utils
from keras.models import load_model

In [2]:
# Download the complete text in ASCII format

from google_drive_downloader import GoogleDriveDownloader as gdd

gdd.download_file_from_google_drive(file_id='1wG4PUnoYVUKrsaWgyWepSacUiYNNDEvM',
                                    dest_path='./wonderland.txt',
                                    unzip=False)

Config GPU RAM

In [3]:
config = tf.compat.v1.ConfigProto()

## Ask for GPU memory gracefully
config.gpu_options.allow_growth = True

sess = tf.compat.v1.Session(config=config)

tf.compat.v1.keras.backend.set_session(sess)

# Read text from the book 

In [4]:
# Load ASCII text and covert to lowercase

filename = "wonderland.txt"
raw_text = open(filename).read()
raw_text = raw_text.lower()

In [5]:
print(raw_text[0:200])

alice's adventures in wonderland

lewis carroll

the millennium fulcrum edition 3.0




chapter i. down the rabbit-hole

alice was beginning to get very tired of sitting by her sister on the
bank, and


In [6]:
chars = sorted(list(set(raw_text)))

print(chars)
print('Number of characters: ',len(chars))

['\n', ' ', '!', '"', "'", '(', ')', '*', ',', '-', '.', '0', '3', ':', ';', '?', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Number of characters:  45


In [7]:
char_to_int = dict((c, i) for i, c in enumerate(chars))
char_to_int['z']

44

INVERSE MAP: get the char from an integer using a *dictionary*  int: char 

In [8]:
int_to_char = dict((i, c) for i, c in enumerate(chars))
int_to_char[44]

'z'

In [9]:
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab)

Total Characters:  144431
Total Vocab:  45


Prediction task

In [10]:
seq_length = 100
dataX=[]

for i in range(0, n_chars - seq_length, 1):
  seq_in = raw_text[i:i+seq_length]
  dataX.append(seq_in)

In [11]:
print('dataX length: ', len(dataX))
print('dataX first training example: \n', dataX[0])

dataX length:  144331
dataX first training example: 
 alice's adventures in wonderland

lewis carroll

the millennium fulcrum edition 3.0




chapter i. d


In [12]:
# The data must be numeric 

seq_length = 100
dataX=[]

for i in range(0, n_chars - seq_length, 1):
  seq_in = raw_text[i:i+seq_length]
  dataX.append([char_to_int[char] for char in seq_in])

In [13]:
print('dataX length: ',len(dataX))
print('dataX first training example: \n',dataX[0])

dataX length:  144331
dataX first training example: 
 [19, 30, 27, 21, 23, 4, 37, 1, 19, 22, 40, 23, 32, 38, 39, 36, 23, 37, 1, 27, 32, 1, 41, 33, 32, 22, 23, 36, 30, 19, 32, 22, 0, 0, 30, 23, 41, 27, 37, 1, 21, 19, 36, 36, 33, 30, 30, 0, 0, 38, 26, 23, 1, 31, 27, 30, 30, 23, 32, 32, 27, 39, 31, 1, 24, 39, 30, 21, 36, 39, 31, 1, 23, 22, 27, 38, 27, 33, 32, 1, 12, 10, 11, 0, 0, 0, 0, 0, 21, 26, 19, 34, 38, 23, 36, 1, 27, 10, 1, 22]


We create the output for each 100 characters windows

In [14]:
seq_length = 100
dataX = []
dataY = []

for i in range(0, n_chars - seq_length, 1):
  
	seq_in = raw_text[i:i + seq_length]
	seq_out = raw_text[i + seq_length]
	dataX.append([char_to_int[char] for char in seq_in])
	dataY.append(char_to_int[seq_out])
  

In [15]:
n_patterns = len(dataX)

print("Total Patterns: ", n_patterns)
print("Pattern shape: ",np.array(dataX).shape)

Total Patterns:  144331
Pattern shape:  (144331, 100)


We see two examples

In [16]:
print("------Window input dataX -------------------------------------")
print("\"", ''.join([int_to_char[value] for value in dataX[201]]), "\"")
print("\n -----Character to predict dataY:")
print("\"", int_to_char[dataY[201]])
print("\n")
print("------Window input dataX -------------------------------------")
print("\"", ''.join([int_to_char[value] for value in dataX[202]]), "\"")
print("\n -----Character to predict dataY:")
print("\"", int_to_char[dataY[202]])
print("\n")
print("\"", ''.join([int_to_char[value] for value in dataY[100:216]]), "\"")

------Window input dataX -------------------------------------
" of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it h "

 -----Character to predict dataY:
" a


------Window input dataX -------------------------------------
" f having nothing to do: once or twice she had peeped into the
book her sister was reading, but it ha "

 -----Character to predict dataY:
" d


"  of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures  "


Prepare our training data to be suitable for use with LSTM in Keras

In [17]:
print('dataX shape', np.array(dataX).shape)

dataX shape (144331, 100)


In [18]:
# Reshape X to be [samples, time steps, features]
X = np.reshape(dataX, (n_patterns, seq_length, 1))

# Normalize
X = X / float(n_vocab)

# One-hot encode the output variable
y = np_utils.to_categorical(dataY)

In [19]:
# or with keras
ykeras=keras.utils.to_categorical(dataY)

print('OHE example numpy',y[3])
print('OHE example keras',ykeras[3])

OHE example numpy [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
OHE example keras [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [20]:
X[200,0:10]*n_vocab

array([[ 1.],
       [33.],
       [24.],
       [ 1.],
       [26.],
       [19.],
       [40.],
       [27.],
       [32.],
       [25.]])

In [21]:
print("\"", ''.join([int_to_char[int(value+0.5)] for value in X[200,:]*n_vocab]), "\"")
print("\"", ''.join([int_to_char[int(value)] for value in dataX[200]]), "\"")

"  of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it  "
"  of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it  "


LSTM model with a single hidden layer with 256 memory units

In [22]:
print('X shape: ', X.shape)
print('y.shape: ', y.shape)
y[0,:]

X shape:  (144331, 100, 1)
y.shape:  (144331, 45)


array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

LSTM Recurrent Neural Network

In [23]:
# Define the LSTM model

model = Sequential()
model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm (LSTM)                  (None, 256)               264192    
_________________________________________________________________
dropout (Dropout)            (None, 256)               0         
_________________________________________________________________
dense (Dense)                (None, 45)                11565     
Total params: 275,757
Trainable params: 275,757
Non-trainable params: 0
_________________________________________________________________


We save the LSTM model into Google Drive

In [24]:
# Define the checkpoint

filepath="/content/drive/MyDrive/Final_Term_Project_Eloise_Jin/Weights_LSTM/weights-improvement-{epoch:02d}-{loss:.4f}.hdf5"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

In [25]:
history = model.fit(X, y, epochs=20, batch_size=128, callbacks=callbacks_list)

Epoch 1/20
1128/1128 [==============================] - 23s 13ms/step - loss: 3.0569

Epoch 00001: loss improved from inf to 2.97668, saving model to /content/drive/MyDrive/Final_Term_Project_Eloise_Jin/Weights_LSTM/weights-improvement-01-2.9767.hdf5
Epoch 2/20
1128/1128 [==============================] - 15s 13ms/step - loss: 2.8056

Epoch 00002: loss improved from 2.97668 to 2.77007, saving model to /content/drive/MyDrive/Final_Term_Project_Eloise_Jin/Weights_LSTM/weights-improvement-02-2.7701.hdf5
Epoch 3/20
1128/1128 [==============================] - 16s 14ms/step - loss: 2.6896

Epoch 00003: loss improved from 2.77007 to 2.66819, saving model to /content/drive/MyDrive/Final_Term_Project_Eloise_Jin/Weights_LSTM/weights-improvement-03-2.6682.hdf5
Epoch 4/20
1128/1128 [==============================] - 16s 14ms/step - loss: 2.6060

Epoch 00004: loss improved from 2.66819 to 2.59076, saving model to /content/drive/MyDrive/Final_Term_Project_Eloise_Jin/Weights_LSTM/weights-improvement

In [26]:
import sys

# Pick a random seed
start = np.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Seed:")
print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")
print('\n GENERATE: \n')

# Generate characters
for i in range(500):
  x = np.reshape(pattern, (1, len(pattern), 1))
  x = x / float(n_vocab)
  prediction = model.predict(x, verbose=0)
  index = np.argmax(prediction)
  result = int_to_char[index]
 
  # Print every ouput character
  sys.stdout.write(result)
  
  # Add output char
  pattern.append(index)
  
  # Remove first char
  pattern = pattern[1:len(pattern)]
  
print("\nDone.")

Seed:
" s she did so, and giving it a violent shake at the end of
every line:

   'speak roughly to your lit "

 GENERATE: 

el that soue

                                                              *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *    *   
Done.


In [27]:
result

' '

# Larger LSTM Recurrent Neural Network

In [28]:
model = Sequential()
model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(256))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm_1 (LSTM)                (None, 100, 256)          264192    
_________________________________________________________________
dropout_1 (Dropout)          (None, 100, 256)          0         
_________________________________________________________________
lstm_2 (LSTM)                (None, 256)               525312    
_________________________________________________________________
dropout_2 (Dropout)          (None, 256)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 45)                11565     
Total params: 801,069
Trainable params: 801,069
Non-trainable params: 0
_________________________________________________________________


In [29]:
filepath="weights-improvement-{epoch:02d}-{loss:.4f}-bigger.hdf5"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

In [30]:
history = model.fit(X, y, epochs=50, batch_size=64, callbacks=callbacks_list)

Epoch 1/50
2256/2256 [==============================] - 48s 20ms/step - loss: 2.9434

Epoch 00001: loss improved from inf to 2.77507, saving model to weights-improvement-01-2.7751-bigger.hdf5
Epoch 2/50
2256/2256 [==============================] - 46s 20ms/step - loss: 2.4739

Epoch 00002: loss improved from 2.77507 to 2.41120, saving model to weights-improvement-02-2.4112-bigger.hdf5
Epoch 3/50
2256/2256 [==============================] - 46s 20ms/step - loss: 2.2377

Epoch 00003: loss improved from 2.41120 to 2.20635, saving model to weights-improvement-03-2.2063-bigger.hdf5
Epoch 4/50
2256/2256 [==============================] - 46s 20ms/step - loss: 2.0943

Epoch 00004: loss improved from 2.20635 to 2.07749, saving model to weights-improvement-04-2.0775-bigger.hdf5
Epoch 5/50
2256/2256 [==============================] - 46s 20ms/step - loss: 1.9905

Epoch 00005: loss improved from 2.07749 to 1.98434, saving model to weights-improvement-05-1.9843-bigger.hdf5
Epoch 6/50
2256/2256 [==

In [ ]:
# Generate characters

for i in range(1000):
	x = np.reshape(pattern, (1, len(pattern), 1))
	x = x / float(n_vocab)
	prediction = model.predict(x, verbose=0)
	index = np.argmax(prediction)
	result = int_to_char[index]
	seq_in = [int_to_char[value] for value in pattern]
	sys.stdout.write(result)
	pattern.append(index)
	pattern = pattern[1:len(pattern)]
  
print ("\nDone.")

 *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

  *

 